# 🧠 Stroke Prediction — End-to-End ML & DL Project

**Dataset:** Healthcare Stroke Dataset (5110 patients)  
**Goal:** Predict whether a patient is likely to get a stroke  
**Pipeline:** EDA → Preprocessing → SMOTE → ML Models → Deep Learning → SHAP Explainability

## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('plots', exist_ok=True)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, roc_curve, confusion_matrix, classification_report)

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import callbacks

import shap
import joblib, json

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
print('All libraries imported!')

## 2️⃣ Load & Explore Data

In [ ]:
df = pd.read_csv('../healthcare-dataset-stroke-data.csv')
df['bmi'] = pd.to_numeric(df['bmi'], errors='coerce')
print(f'Shape: {df.shape}')
print(f'Stroke rate: {df["stroke"].mean()*100:.2f}%')
df.head()

In [ ]:
print('--- MISSING VALUES ---')
print(df.isnull().sum())
print('\n--- DATA TYPES ---')
print(df.dtypes)
print('\n--- STATS ---')
df.describe()

## 3️⃣ Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['stroke'].value_counts()
axes[0].pie(counts, labels=['No Stroke','Stroke'], colors=['#2ecc71','#e74c3c'],
            autopct='%1.1f%%', startangle=90, explode=(0, 0.1), shadow=True)
axes[0].set_title('Target Distribution', fontsize=14, fontweight='bold')
axes[1].bar(['No Stroke','Stroke'], counts, color=['#2ecc71','#e74c3c'], width=0.5)
axes[1].set_title('Class Counts', fontsize=14, fontweight='bold')
for bar, c in zip(axes[1].patches, counts):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 f'{c}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/target_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(df['age'], bins=40, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Age Distribution', fontsize=13, fontweight='bold')
df[df['stroke']==0]['age'].plot(kind='hist', bins=40, alpha=0.7, color='#2ecc71', label='No Stroke', ax=axes[1])
df[df['stroke']==1]['age'].plot(kind='hist', bins=40, alpha=0.7, color='#e74c3c', label='Stroke', ax=axes[1])
axes[1].set_title('Age by Stroke Status', fontsize=13, fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('plots/age_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
num_features = ['age', 'avg_glucose_level', 'bmi']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(num_features):
    data = [df[df['stroke']==0][feat].dropna(), df[df['stroke']==1][feat].dropna()]
    bp = axes[i].boxplot(data, patch_artist=True, medianprops=dict(color='white', linewidth=2))
    for patch, c in zip(bp['boxes'], ['#2ecc71','#e74c3c']):
        patch.set_facecolor(c); patch.set_alpha(0.8)
    axes[i].set_title(feat.replace('_',' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_xticklabels(['No Stroke','Stroke'])
plt.suptitle('Numerical Features vs Stroke', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
cat_features = ['gender','hypertension','heart_disease','ever_married','work_type','Residence_type','smoking_status']
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()
palette = ['#3498db','#e74c3c','#9b59b6','#f39c12','#1abc9c','#e67e22']
for i, feat in enumerate(cat_features):
    stroke_rate = df.groupby(feat)['stroke'].mean() * 100
    stroke_rate.plot(kind='bar', ax=axes[i], color=palette[:len(stroke_rate)], edgecolor='white', width=0.6)
    axes[i].set_title(feat.replace('_',' ').title(), fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Stroke Rate (%)')
    axes[i].tick_params(axis='x', rotation=30)
axes[-1].set_visible(False)
plt.suptitle('Stroke Rate by Categorical Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/categorical_stroke_rate.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size':11})
ax.set_title('Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc = df['stroke'].map({0:'#2ecc71',1:'#e74c3c'})
axes[0].scatter(df['age'], df['avg_glucose_level'], c=sc, alpha=0.5, s=20)
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Avg Glucose Level')
axes[0].set_title('Age vs Glucose by Stroke', fontsize=13, fontweight='bold')
green_p = mpatches.Patch(color='#2ecc71', label='No Stroke')
red_p   = mpatches.Patch(color='#e74c3c', label='Stroke')
axes[0].legend(handles=[green_p, red_p])
axes[1].scatter(df['age'], df['bmi'], c=sc, alpha=0.5, s=20)
axes[1].set_xlabel('Age'); axes[1].set_ylabel('BMI')
axes[1].set_title('Age vs BMI by Stroke', fontsize=13, fontweight='bold')
axes[1].legend(handles=[green_p, red_p])
plt.tight_layout()
plt.savefig('plots/scatter_plots.png', bbox_inches='tight')
plt.show()

## 4️⃣ Preprocessing & Feature Engineering

In [ ]:
df_clean = df.drop(columns=['id']).copy()
df_clean = df_clean[df_clean['gender'] != 'Other']
df_clean['bmi'] = df_clean['bmi'].fillna(df_clean['bmi'].median())

# Feature Engineering
df_clean['age_group'] = pd.cut(df_clean['age'],
    bins=[0,12,17,35,55,65,120],
    labels=['Child','Teen','YoungAdult','MiddleAge','Senior','Elderly'])

df_clean['risk_score'] = (df_clean['hypertension'] +
    df_clean['heart_disease'] +
    (df_clean['avg_glucose_level'] > 140).astype(int) +
    (df_clean['bmi'] > 30).astype(int) +
    (df_clean['age'] > 60).astype(int))

df_clean['glucose_cat'] = pd.cut(df_clean['avg_glucose_level'],
    bins=[0,70,100,125,200,1000],
    labels=['Low','Normal','Prediabetic','Diabetic','VeryHigh'])

df_clean['bmi_cat'] = pd.cut(df_clean['bmi'],
    bins=[0,18.5,25,30,100],
    labels=['Underweight','Normal','Overweight','Obese'])

print(f'Feature engineering done! Shape: {df_clean.shape}')

In [ ]:
df_enc = df_clean.copy()
df_enc['gender']         = LabelEncoder().fit_transform(df_enc['gender'])
df_enc['ever_married']   = df_enc['ever_married'].map({'Yes':1,'No':0})
df_enc['Residence_type'] = df_enc['Residence_type'].map({'Urban':1,'Rural':0})
df_enc = pd.get_dummies(df_enc,
    columns=['work_type','smoking_status','age_group','glucose_cat','bmi_cat'],
    drop_first=True)
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)
print(f'Encoded shape: {df_enc.shape}')
df_enc.head()

In [ ]:
X = df_enc.drop(columns=['stroke'])
y = df_enc['stroke']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

scaler = RobustScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 5️⃣ Handle Class Imbalance (SMOTE)

In [ ]:
print(f'Before SMOTE - No Stroke: {(y_train==0).sum()} | Stroke: {(y_train==1).sum()}')
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_train_sc, y_train)
print(f'After  SMOTE - No Stroke: {(y_res==0).sum()} | Stroke: {(y_res==1).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in zip(axes, [y_train, y_res], ['Before SMOTE','After SMOTE']):
    vals = pd.Series(data).value_counts().sort_index()
    ax.bar(['No Stroke','Stroke'], vals, color=['#2ecc71','#e74c3c'], edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    for i, v in enumerate(vals): ax.text(i, v+10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/smote_balance.png', bbox_inches='tight')
plt.show()

## 6️⃣ Machine Learning Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=RANDOM_STATE),
    'KNN':                 KNeighborsClassifier(n_neighbors=7),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, scale_pos_weight=19, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
    'SVM':                 SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=RANDOM_STATE)
}

results = {}
for name, model in models.items():
    print(f'Training: {name}...')
    model.fit(X_res, y_res)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:,1]
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'F1 Score':  round(f1_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4),
        'y_pred': y_pred, 'y_prob': y_prob, 'model': model
    }
    print(f'  AUC={results[name]["ROC-AUC"]:.4f} | Recall={results[name]["Recall"]:.4f}')
print('All ML models done!')

In [ ]:
metrics_df = pd.DataFrame({k: {m:v for m,v in v.items() if m not in ['y_pred','y_prob','model']}
    for k,v in results.items()}).T.sort_values('ROC-AUC', ascending=False)
print('=== MODEL COMPARISON ===')
print(metrics_df.to_string())
metrics_df.style.background_gradient(cmap='YlOrRd').format('{:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, lw=2.5, label=f'{name} (AUC={res["ROC-AUC"]:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=15, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig('plots/roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
n = len(results)
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()
for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
                xticklabels=['No Stroke','Stroke'],
                yticklabels=['No Stroke','Stroke'], ax=axes[i],
                linewidths=0.5, annot_kws={'size':13,'weight':'bold'})
    axes[i].set_title(f'{name}\nAUC:{res["ROC-AUC"]:.3f}', fontsize=10, fontweight='bold')
axes[-1].set_visible(False)
plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', bbox_inches='tight')
plt.show()

## 7️⃣ Deep Learning — Neural Network (Keras)

In [ ]:
def build_model(input_dim):
    model = Sequential([
        Dense(256, input_dim=input_dim, activation='relu'),
        BatchNormalization(), Dropout(0.4),
        Dense(128, activation='relu'),
        BatchNormalization(), Dropout(0.35),
        Dense(64, activation='relu'),
        BatchNormalization(), Dropout(0.3),
        Dense(32, activation='relu'), Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model

nn = build_model(X_res.shape[1])
nn.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = dict(enumerate(cw))

early_stop = callbacks.EarlyStopping(monitor='val_auc', patience=15, restore_best_weights=True, mode='max')
lr_reduce  = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=8, min_lr=1e-6)

history = nn.fit(
    X_res, y_res,
    validation_data=(X_test_sc, y_test),
    epochs=100, batch_size=64,
    callbacks=[early_stop, lr_reduce],
    class_weight=cw_dict, verbose=1
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (tm, vm, title, col) in zip(axes.flatten(), [
    ('loss','val_loss','Loss','#e74c3c'),
    ('accuracy','val_accuracy','Accuracy','#3498db'),
    ('auc','val_auc','AUC-ROC','#2ecc71'),
    ('recall','val_recall','Recall','#f39c12')]):
    ax.plot(history.history[tm], color=col, lw=2.5, label='Training')
    ax.plot(history.history[vm], color=col, lw=2.5, linestyle='--', alpha=0.7, label='Validation')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
plt.suptitle('Neural Network Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/nn_training_history.png', bbox_inches='tight')
plt.show()

In [ ]:
y_prob_nn = nn.predict(X_test_sc).flatten()
y_pred_nn = (y_prob_nn >= 0.4).astype(int)

print('=== NEURAL NETWORK RESULTS ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_nn):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_nn):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_nn):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_nn):.4f}')
print()
print(classification_report(y_test, y_pred_nn, target_names=['No Stroke','Stroke']))

results['Neural Network'] = {
    'Accuracy':  round(accuracy_score(y_test, y_pred_nn), 4),
    'F1 Score':  round(f1_score(y_test, y_pred_nn), 4),
    'Precision': round(precision_score(y_test, y_pred_nn, zero_division=0), 4),
    'Recall':    round(recall_score(y_test, y_pred_nn), 4),
    'ROC-AUC':   round(roc_auc_score(y_test, y_prob_nn), 4),
    'y_pred': y_pred_nn, 'y_prob': y_prob_nn, 'model': nn
}

## 8️⃣ Final Model Comparison

In [ ]:
all_metrics = pd.DataFrame({k: {m:v for m,v in v.items() if m not in ['y_pred','y_prob','model']}
    for k,v in results.items()}).T.astype(float).sort_values('ROC-AUC', ascending=False)

print('=== FINAL LEADERBOARD ===')
print(all_metrics.to_string())

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(all_metrics))
w = 0.15
cols = ['#3498db','#2ecc71','#f39c12','#e74c3c','#9b59b6']
for i, (metric, color) in enumerate(zip(['Accuracy','F1 Score','Precision','Recall','ROC-AUC'], cols)):
    ax.bar(x + i*w, all_metrics[metric].astype(float), w, label=metric, color=color, alpha=0.85)
ax.set_xticks(x + w*2)
ax.set_xticklabels(all_metrics.index, rotation=25, ha='right')
ax.set_title('All Models — Final Comparison', fontsize=15, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('plots/final_comparison.png', bbox_inches='tight')
plt.show()

## 9️⃣ SHAP Explainability

In [ ]:
feature_names = list(X.columns)
best_name = all_metrics.index[0] if all_metrics.index[0] != 'Neural Network' else all_metrics.index[1]
best_model = results[best_name]['model']
print(f'Best model for SHAP: {best_name}')

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_sc)
if isinstance(shap_values, list): shap_values = shap_values[1]
print('SHAP values computed!')

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sc, feature_names=feature_names, show=False, max_display=15)
plt.title(f'SHAP Feature Importance — {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/shap_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values, X_test_sc, feature_names=feature_names,
                  plot_type='bar', show=False, max_display=15)
plt.title('SHAP Feature Impact (Mean |SHAP value|)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/shap_bar.png', bbox_inches='tight')
plt.show()

In [ ]:
stroke_idx = np.where(y_test.values == 1)[0][0]
expected_val = explainer.expected_value
if isinstance(expected_val, list): expected_val = expected_val[1]

shap_exp = shap.Explanation(
    values=shap_values[stroke_idx],
    base_values=expected_val,
    data=X_test_sc[stroke_idx],
    feature_names=feature_names
)
plt.figure(figsize=(14, 7))
shap.plots.waterfall(shap_exp, max_display=12, show=False)
plt.title('SHAP Waterfall — Individual High-Risk Patient', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/shap_waterfall.png', bbox_inches='tight')
plt.show()

## 🔟 Save Models

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/best_ml_model.pkl')
joblib.dump(scaler,     'models/scaler.pkl')
with open('models/feature_names.json', 'w') as f:
    json.dump(feature_names, f)
nn.save('models/stroke_nn_model.keras')

print('Models saved!')
print(f'Best Model: {best_name}')
print(f'  ROC-AUC: {all_metrics.loc[best_name, "ROC-AUC"]:.4f}')
print(f'  Recall:  {all_metrics.loc[best_name, "Recall"]:.4f}')

mean_shap = np.abs(shap_values).mean(axis=0)
top5 = pd.Series(mean_shap, index=feature_names).sort_values(ascending=False).head(5)
print('\nTop 5 Risk Factors:')
for feat, val in top5.items():
    print(f'  {feat}: {val:.4f}')